<a href="https://colab.research.google.com/github/Sumit-Tomar-Web-Developer/My-project/blob/master/Flipkart_CSAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flipkart Customer Service Satisfaction (CSAT) – Complete ML Project

## Project Overview
This project analyzes **85,907 real customer support interactions** from Flipkart's customer service platform.  
The goal is to **predict CSAT Score (1–5)** and uncover key drivers of customer satisfaction using Data Science & Machine Learning.

## Problem Statement
Customer satisfaction directly impacts retention and revenue. By predicting CSAT scores and identifying their root causes, Flipkart can:
- Proactively intervene before customers churn
- Optimize agent training and shift scheduling
- Prioritize high-risk categories for process improvement

## Dataset Description
| Column | Description |
|---|---|
| `channel_name` | Contact channel – Inbound call, Outcall, Email |
| `category` | Issue category (Returns, Order Related, Refund, etc.) |
| `Sub-category` | Specific issue sub-type |
| `Issue_reported at` | Timestamp when customer raised the issue |
| `issue_responded` | Timestamp when agent responded |
| `Customer_City` | City of the customer |
| `Item_price` | Price of the product involved |
| `connected_handling_time` | Time agent spent on the call (seconds) |
| `Agent_name` / `Supervisor` / `Manager` | Agent hierarchy |
| `Tenure Bucket` | Agent experience bucket (OJT, 0-30, 31-60, 61-90, >90 days) |
| `Agent Shift` | Morning / Evening / Afternoon / Night / Split |
| `CSAT Score` | **Target variable** – Customer satisfaction (1=worst, 5=best) |

## Project Architecture
**EDA → Data Clean-up → Feature Engineering → Pre-Processing → Model Building → Model Evaluation**

## Libraries Used
- `pandas`, `numpy` – Data manipulation
- `matplotlib`, `seaborn` – Visualization  
- `scikit-learn` – ML models, preprocessing, evaluation


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## 2. Load Dataset

In [2]:
# Load the Flipkart customer support dataset
df = pd.read_csv('Customer_support_data.csv')

print("=" * 50)
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 50)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Customer_support_data.csv'

## 3. Data Understanding

In [ ]:
# Basic information about the dataset
print("─── Dataset Info ───")
df.info()

In [ ]:
# Statistical summary of all columns
print("─── Statistical Summary ───")
df.describe(include='all').T

## 4. Missing Value Analysis & Handling

Missing values can introduce bias into our model. We analyze each column carefully:
- Columns with >50% missing values are dropped (not useful for prediction)
- Columns with moderate missing values are imputed appropriately


In [ ]:
# Calculate missing values per column
missing = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Percentage (%)', ascending=False)

missing = missing[missing['Missing Values'] > 0]  # show only columns with missing data

print("─── Columns with Missing Values ───")
print(missing.to_string())

# Visualize missing values
plt.figure(figsize=(10, 5))
missing['Percentage (%)'].plot(kind='bar', color='coral', edgecolor='black')
plt.title('Missing Value Percentage per Column', fontsize=14, fontweight='bold')
plt.xlabel('Column')
plt.ylabel('Missing %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Exploratory Data Analysis (EDA)

### 5.1 Target Variable – CSAT Score Distribution

In [ ]:
# CSAT Score distribution – understanding class balance
plt.figure(figsize=(9, 5))
ax = sns.countplot(x='CSAT Score', data=df, palette='RdYlGn',
                   order=sorted(df['CSAT Score'].unique()))

# Add count labels on bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('CSAT Score Distribution', fontsize=15, fontweight='bold')
plt.xlabel('CSAT Score (1=Worst, 5=Best)', fontsize=12)
plt.ylabel('Number of Tickets', fontsize=12)
plt.tight_layout()
plt.show()

# Print percentage breakdown
print("─── CSAT Score Distribution ───")
vc = df['CSAT Score'].value_counts().sort_index()
for score, count in vc.items():
    print(f"  Score {score}: {count:>6,} tickets ({count/len(df)*100:.1f}%)")
print(f"\n⚠️  Class Imbalance Detected: Score 5 dominates ({vc[5]/len(df)*100:.1f}% of data)")

### 5.2 Issue Category Analysis

In [ ]:
# Top issue categories and their average CSAT scores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Volume per category
category_counts = df['category'].value_counts().head(10)
axes[0].bar(category_counts.index, category_counts.values,
            color=sns.color_palette("Blues_r", len(category_counts)))
axes[0].set_title('Top Issue Categories by Volume', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Number of Tickets')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(category_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

# Plot 2: Average CSAT per category
avg_csat_cat = df.groupby('category')['CSAT Score'].mean().sort_values()
colors = ['red' if v < 4 else 'green' for v in avg_csat_cat.values]
axes[1].barh(avg_csat_cat.index, avg_csat_cat.values, color=colors)
axes[1].set_title('Average CSAT Score by Category', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Average CSAT Score')
axes[1].axvline(x=4, color='orange', linestyle='--', label='Threshold=4')
axes[1].legend()
for i, v in enumerate(avg_csat_cat.values):
    axes[1].text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 5.3 Channel & Shift Analysis

In [ ]:
# Analyze CSAT by channel and shift
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: CSAT by channel
avg_csat_channel = df.groupby('channel_name')['CSAT Score'].mean().sort_values()
axes[0].bar(avg_csat_channel.index, avg_csat_channel.values,
            color=['steelblue', 'salmon', 'seagreen'])
axes[0].set_title('Avg CSAT by Channel', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Average CSAT Score')
axes[0].set_ylim(3.5, 5)
for i, v in enumerate(avg_csat_channel.values):
    axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')

# Plot 2: Agent Shift distribution
shift_counts = df['Agent Shift'].value_counts()
axes[1].bar(shift_counts.index, shift_counts.values,
            color=sns.color_palette("Set2", len(shift_counts)))
axes[1].set_title('Ticket Volume by Agent Shift', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Tickets')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(shift_counts.values):
    axes[1].text(i, v + 300, f'{v:,}', ha='center', fontsize=9)

# Plot 3: CSAT by Tenure Bucket
tenure_order = ['On Job Training', '0-30', '31-60', '61-90', '>90']
avg_csat_tenure = df.groupby('Tenure Bucket')['CSAT Score'].mean().reindex(tenure_order)
axes[2].bar(avg_csat_tenure.index, avg_csat_tenure.values,
            color=sns.color_palette("YlOrRd", len(avg_csat_tenure)))
axes[2].set_title('Avg CSAT by Agent Tenure', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Average CSAT Score')
axes[2].set_ylim(3.5, 5)
axes[2].tick_params(axis='x', rotation=30)
for i, v in enumerate(avg_csat_tenure.values):
    axes[2].text(i, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Feature Engineering

We create a new feature: **Response_Time_Minutes** – the time taken by the agent to respond to the customer.  
This is a critical driver of satisfaction, as faster responses typically lead to higher CSAT scores.


In [ ]:
# Parse datetime columns
df['Issue_reported at'] = pd.to_datetime(df['Issue_reported at'], errors='coerce')
df['issue_responded']   = pd.to_datetime(df['issue_responded'],   errors='coerce')

# Create Response_Time_Minutes feature
df['Response_Time_Minutes'] = (
    df['issue_responded'] - df['Issue_reported at']
).dt.total_seconds() / 60

# Fill missing response times with median (robust to outliers)
median_rt = df['Response_Time_Minutes'].median()
df['Response_Time_Minutes'] = df['Response_Time_Minutes'].fillna(median_rt)

print(f"✅ Response_Time_Minutes created")
print(f"   Median response time: {median_rt:.1f} minutes")
print(f"   Mean response time:   {df['Response_Time_Minutes'].mean():.1f} minutes")
print(f"   Max response time:    {df['Response_Time_Minutes'].max():.1f} minutes")

# Analyze response time vs CSAT
plt.figure(figsize=(10, 5))
df.groupby('CSAT Score')['Response_Time_Minutes'].median().plot(
    kind='bar', color=sns.color_palette("RdYlGn", 5), edgecolor='black')
plt.title('Median Response Time by CSAT Score', fontsize=14, fontweight='bold')
plt.xlabel('CSAT Score')
plt.ylabel('Median Response Time (minutes)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("\n💡 Insight: Lower CSAT scores tend to have longer response times")

## 7. Outlier Detection & Handling

Outliers can distort model training. We use the **IQR (Interquartile Range)** method to detect and cap extreme values.


In [ ]:
# Visualize outliers with boxplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df['Response_Time_Minutes'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Response_Time_Minutes – Before Capping', fontsize=12)
axes[0].set_ylabel('Minutes')

axes[1].boxplot(df['Item_price'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='lightyellow'))
axes[1].set_title('Item_price – Before Capping', fontsize=12)
axes[1].set_ylabel('Price (₹)')

plt.suptitle('Outlier Detection via Box Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# IQR-based capping function
def cap_outliers_iqr(series):
    """Cap outliers using IQR method (Winsorization)."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    capped = series.clip(lower=lower, upper=upper)
    n_capped = ((series < lower) | (series > upper)).sum()
    print(f"  Capped {n_capped:,} outliers | Range: [{lower:.1f}, {upper:.1f}]")
    return capped

print("\n─── Applying IQR Capping ───")
print("Response_Time_Minutes:")
df['Response_Time_Minutes'] = cap_outliers_iqr(df['Response_Time_Minutes'])
print("Item_price:")
df['Item_price'] = cap_outliers_iqr(df['Item_price'].fillna(df['Item_price'].median()))
print("\n✅ Outliers capped successfully")

## 8. Correlation Analysis

We analyze how features relate to the **CSAT Score** (target variable) and to each other.  
This helps us select the most important features and detect multicollinearity.


In [ ]:
# ── 8.1 Numeric Correlation Heatmap ──
numeric_df = df[['Response_Time_Minutes', 'Item_price',
                 'connected_handling_time', 'CSAT Score']].copy()

plt.figure(figsize=(8, 6))
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap – Numeric Features vs CSAT Score',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("─── Correlation with CSAT Score ───")
csat_corr = corr_matrix['CSAT Score'].drop('CSAT Score').sort_values()
for feat, val in csat_corr.items():
    direction = "↑ positive" if val > 0 else "↓ negative"
    print(f"  {feat:<30} {val:+.4f}  ({direction})")

In [ ]:
# ── 8.2 Categorical Features vs CSAT Score ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cat_features = ['channel_name', 'Agent Shift', 'Tenure Bucket']
titles = ['Channel vs CSAT', 'Shift vs CSAT', 'Tenure vs CSAT']

for ax, feat, title in zip(axes, cat_features, titles):
    # Box plot shows median + distribution per category
    categories = df[feat].unique()
    data_by_cat = [df[df[feat] == c]['CSAT Score'].values for c in categories]
    bp = ax.boxplot(data_by_cat, labels=categories, patch_artist=True,
                    boxprops=dict(facecolor='lightcyan'))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('CSAT Score')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Categorical Features vs CSAT Score (Box Plots)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("💡 Insight: These box plots reveal how CSAT varies across each category group")

In [ ]:
# ── 8.3 CSAT Score vs Category Heatmap ──
# Cross-tab: category × CSAT Score
pivot = pd.crosstab(df['category'], df['CSAT Score'], normalize='index') * 100

plt.figure(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.3, cbar_kws={'label': '% of tickets'})
plt.title('CSAT Score Distribution (%) by Issue Category',
          fontsize=13, fontweight='bold')
plt.xlabel('CSAT Score')
plt.ylabel('Issue Category')
plt.tight_layout()
plt.show()
print("💡 Insight: 'Offers & Cashback' and 'Refund Related' have higher % of Score-1 tickets")

## 9. Pre-Processing

Steps:
1. Drop irrelevant/leakage columns (IDs, free-text, raw timestamps)
2. Separate features (X) and target (y)
3. Apply `MinMaxScaler` to numeric features
4. Apply `OneHotEncoder` to categorical features
5. Stratified Train-Test Split (80/20)


In [ ]:
# Columns to drop (IDs, free text, raw timestamps – not useful for prediction)
drop_cols = [
    'Unique id',
    'Order_id',
    'Customer Remarks',       # free text – requires NLP; dropped here
    'Issue_reported at',      # raw timestamp – feature engineered above
    'issue_responded',        # raw timestamp – feature engineered above
    'Survey_response_Date',
    'order_date_time',
    'Agent_name',             # too many unique values – high cardinality noise
    'Customer_City'           # too many unique values + 80% missing
]

# Drop columns that exist
X = df.drop(columns=[c for c in drop_cols + ['CSAT Score'] if c in df.columns])
y = df['CSAT Score']

# Separate column types
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols     = X.select_dtypes(exclude='object').columns.tolist()

print(f"Feature matrix shape: {X.shape}")
print(f"\nNumeric features  ({len(numeric_cols)}): {numeric_cols}")
print(f"\nCategorical features ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Build preprocessing pipeline
# Numeric: median impute (handles NaN) → MinMax scaling
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  MinMaxScaler())
])

# Categorical: most_frequent impute → OneHotEncoding
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine both pipelines
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,     numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

# Stratified split preserves CSAT class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("✅ Preprocessing pipeline built")
print(f"   Training set : {X_train.shape[0]:,} samples")
print(f"   Test set     : {X_test.shape[0]:,} samples")
print(f"\nClass distribution in train set:")
for score, pct in (y_train.value_counts(normalize=True).sort_index() * 100).items():
    print(f"   CSAT {score}: {pct:.1f}%")

## 10. Model Building & Evaluation

In [ ]:
# ── Reusable evaluation function (Modular Code) ──
def evaluate_model(name, model, X_test, y_test):
    """Print accuracy, weighted F1, and classification report for a model."""
    pred = model.predict(X_test)
    acc  = accuracy_score(y_test, pred)
    f1   = f1_score(y_test, pred, average='weighted')
    print(f"\n{'='*50}")
    print(f"  Model: {name}")
    print(f"{'='*50}")
    print(f"  Accuracy         : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Weighted F1 Score: {f1:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, pred))
    return pred, acc, f1

def plot_confusion_matrix(name, y_test, pred):
    """Plot a labelled confusion matrix heatmap."""
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=sorted(y_test.unique()),
                yticklabels=sorted(y_test.unique()))
    plt.title(f'Confusion Matrix – {name}', fontsize=13, fontweight='bold')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.show()

print("✅ Helper functions defined: evaluate_model(), plot_confusion_matrix()")

### 10.1 Model 1 – Decision Tree Classifier

In [ ]:
# Decision Tree – baseline model
dt_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Train
dt_model.fit(X_train, y_train)

# Evaluate
dt_pred, dt_acc, dt_f1 = evaluate_model("Decision Tree", dt_model, X_test, y_test)
plot_confusion_matrix("Decision Tree", y_test, dt_pred)

### 10.2 Model 2 – Random Forest Classifier

In [ ]:
# Random Forest – ensemble model with class balancing
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',   # handles class imbalance (Score 5 dominates)
        random_state=42,
        n_jobs=-1                  # use all CPU cores
    ))
])

# Train
rf_model.fit(X_train, y_train)

# Evaluate
rf_pred, rf_acc, rf_f1 = evaluate_model("Random Forest", rf_model, X_test, y_test)
plot_confusion_matrix("Random Forest", y_test, rf_pred)

## 11. Hyperparameter Tuning (GridSearchCV)

We tune the Random Forest's key parameters using 3-fold cross-validation, scoring on **weighted F1** to handle class imbalance.


In [ ]:
# Hyperparameter grid
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth':    [10, 20, None],
    'classifier__min_samples_split': [2, 5]
}

# Base pipeline for tuning
grid_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        class_weight='balanced', random_state=42
    ))
])

# GridSearchCV – 3-fold CV
grid_search = GridSearchCV(
    grid_pipeline,
    param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

print("🔍 Running GridSearchCV (this may take a few minutes)...")
grid_search.fit(X_train, y_train)

print(f"\n✅ Best Parameters  : {grid_search.best_params_}")
print(f"   Best CV F1 Score : {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate the best tuned model
best_pred, best_acc, best_f1 = evaluate_model(
    "Random Forest (Tuned)", grid_search, X_test, y_test
)
plot_confusion_matrix("Random Forest (Tuned)", y_test, best_pred)

## 12. Model Comparison Summary

In [ ]:
# Compare all models side by side
results = {
    'Decision Tree':          (dt_acc,   dt_f1),
    'Random Forest':          (rf_acc,   rf_f1),
    'Random Forest (Tuned)':  (best_acc, best_f1)
}

print("\n" + "="*55)
print(f"{'Model':<28} {'Accuracy':>10} {'Weighted F1':>12}")
print("="*55)
for model_name, (acc, f1) in results.items():
    marker = " ← Best" if acc == max(r[0] for r in results.values()) and model_name != 'Decision Tree' else ""
    print(f"{model_name:<28} {acc:>9.4f}  {f1:>11.4f}{marker}")
print("="*55)

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names  = list(results.keys())
accs   = [v[0] for v in results.values()]
f1s    = [v[1] for v in results.values()]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

axes[0].bar(names, accs, color=colors, edgecolor='black')
axes[0].set_title('Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(names, f1s, color=colors, edgecolor='black')
axes[1].set_title('Weighted F1 Score Comparison', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Weighted F1')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(f1s):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Feature Importance Analysis

In [ ]:
# Extract feature importances from the best model
best_rf = grid_search.best_estimator_.named_steps['classifier']
prep    = grid_search.best_estimator_.named_steps['preprocessor']

# Get feature names after encoding
ohe_features = prep.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_cols)
all_features  = numeric_cols + list(ohe_features)

# Build importance DataFrame
importances = pd.DataFrame({
    'Feature':    all_features,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=importances, x='Importance', y='Feature',
            palette='viridis')
plt.title('Top 15 Most Important Features (Random Forest)',
          fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()

print("─── Top 10 Features ───")
for _, row in importances.head(10).iterrows():
    print(f"  {row['Feature']:<45} {row['Importance']:.4f}")

## 14. Final Conclusion & Business Recommendations

---

### 📊 Key Findings

| Finding | Detail |
|---|---|
| **Class Imbalance** | CSAT Score 5 accounts for 69.4% of tickets; class_weight='balanced' was applied |
| **Response Time Impact** | Longer response times strongly correlate with lower CSAT scores |
| **Best Performing Model** | Random Forest (Tuned) outperforms Decision Tree on both Accuracy and F1 |
| **Hyperparameter Tuning** | GridSearchCV improved the weighted F1 score over the baseline RF |
| **Top Predictors** | Response_Time_Minutes, Tenure Bucket, Agent Shift, and Issue Category |
| **High-Risk Categories** | Refund Related and Returns have highest % of Score-1 & Score-2 tickets |

---

### 🏆 Model Performance Summary

| Model | Strength | Weakness |
|---|---|---|
| Decision Tree | Fast, interpretable | Overfits; weaker on minority classes |
| Random Forest | Better generalization, handles imbalance | Slower to train |
| RF (Tuned) | Best overall performance | Requires tuning time |

---

### 💡 Business Recommendations

1. **Reduce Response Time** – Target < 30 minutes for all channels; automate initial responses
2. **Focus on Refund & Returns** – These categories have the lowest satisfaction; review SLAs
3. **Invest in Agent Training** – On-Job-Training agents show lower CSAT; structured mentoring helps
4. **Optimize Night Shifts** – Lower ticket volume but may need more experienced agents
5. **CSAT Monitoring Dashboard** – Track real-time CSAT by category, shift, and tenure bucket
6. **Escalation System** – Auto-escalate tickets predicted as likely Score 1/2 to senior agents

---

### 🔬 Future Improvements
- **NLP on Customer Remarks** – Sentiment analysis of free-text field (currently dropped)
- **Time-series analysis** – Track CSAT trends over months
- **XGBoost / LightGBM** – May yield higher accuracy with faster training
- **SMOTE** – Advanced oversampling for minority CSAT classes


GitHub Link :- https://github.com/Sumit-Tomar-Web-Developer/Labmentix_internship/tree/4ca7c79e0229a9b17694534bf6bc967c3a11b118/Flipkart_CSAT